# Physician Drug Adoption Prediction — ABC Pharma / Drug XYZ
### ML 102B Capstone — Predicting Next-Quarter Brand Adoption Among Non-Adopting Physicians

**Business Objective**

ABC Pharma launched drug **XYZ** (treatment for multiple indications of stage-1 chronic kidney disease) ~2.5 years ago.
Many physicians have adopted (prescribed) it; some have never prescribed it ("non-adopters").

The business goal is: **for physicians who are currently non-adopters, predict the probability that they will adopt
(prescribe XYZ for the first time) in the *next* quarter**, so that marketing/sales effort — which is capacity-constrained —
can be prioritized toward the physicians most likely to convert.

The final business deliverable is a ranked list of **~1,500 Quarter-11 non-adopting physicians**, prioritized by predicted
adoption probability, with the **top 20%** flagged as High Priority for sales targeting.

**Primary metric:** `Lift@20%` (NOT accuracy — the classes are imbalanced and the business only cares about the top slice
it can act on). Secondary metrics: ROC-AUC, PR-AUC, Precision, Recall, F1, Confusion Matrix.

**⚠️ Data-mapping caveats (please verify before trusting the pipeline end-to-end):**
- `Input_data_file2.csv` static-profile column *names* could not be reliably read from the provided screenshot (filter
  dropdown arrows truncate the header text). This notebook therefore reads file2's actual header row at runtime and
  auto-classifies columns as categorical/numeric rather than hardcoding guessed names — inspect the Cell 4 printout.
- `new_prescriptions` in file1 is ambiguous (brand-specific vs. market-wide) and is flagged as a **possible leakage
  risk** — its correlation with the target is checked explicitly in the EDA/leakage-audit section before it is kept.


In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                              recall_score, f1_score, confusion_matrix)
from sklearn.inspection import permutation_importance
from statsmodels.stats.outliers_influence import variance_inflation_factor

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("xgboost not installed — install with `pip install xgboost` before running the XGBoost cells.")

pd.set_option("display.max_columns", 100)
sns.set_style("whitegrid")
RANDOM_STATE = 42


In [ ]:
# ------------------------------------------------------------------
# File paths — adjust if your files live elsewhere
# ------------------------------------------------------------------
FILE1_PATH = "Input_data_file1.csv"     # quarterly physician activity (10 quarters, ~10,000 physicians)
FILE2_PATH = "Input_data_file2.csv"     # static physician profile (~10,000 physicians)
TEST_PATH  = "Test_physicians.csv"      # ~1,502 non-adopted physicians needing Q11 prediction

df1 = pd.read_csv(FILE1_PATH)
df2 = pd.read_csv(FILE2_PATH)
df_test = pd.read_csv(TEST_PATH)

print("Input_data_file1 shape:", df1.shape)
print("Input_data_file2 shape:", df2.shape)
print("Test_physicians shape:", df_test.shape)


In [ ]:
# ------------------------------------------------------------------
# Inspect datasets
# ------------------------------------------------------------------
print("=== Input_data_file1 ===")
print(df1.dtypes)
print(df1.head())
print("Unique physicians:", df1['physician_id'].nunique())
print("Duplicate physician_id+year_quarter rows:", df1.duplicated(subset=['physician_id','year_quarter']).sum())
print("Missing values per column:\n", df1.isna().sum())

print("\n=== Input_data_file2 ===")
print("Columns (READ CAREFULLY — header names could not be verified from screenshots):")
print(list(df2.columns))
print(df2.dtypes)
print(df2.head())
print("Unique physicians:", df2['physician_id'].nunique())
print("Duplicate physician_id rows:", df2.duplicated(subset=['physician_id']).sum())
print("Missing values per column:\n", df2.isna().sum())

print("\n=== Test_physicians ===")
print(df_test.dtypes)
print(df_test.head())
print("Unique physicians:", df_test['physician_id'].nunique())

# Auto-classify Input_data_file2 static columns (excluding the ID) into categorical vs numeric
# This avoids hardcoding column names we could not reliably read from the screenshots.
static_id_col = 'physician_id'
static_cat_cols = [c for c in df2.columns if c != static_id_col and df2[c].dtype == object]
static_num_cols = [c for c in df2.columns if c != static_id_col and c not in static_cat_cols]
print("\nAuto-detected static CATEGORICAL columns:", static_cat_cols)
print("Auto-detected static NUMERIC columns:", static_num_cols)


### Merge Input 1 and Input 2

We merge on `physician_id` (a **many-to-one** merge: many quarterly rows in file1 map to one static profile row
in file2). This is necessary because the model needs both **time-varying behavior** (visits, samples, digital
engagement, etc.) and **static physician characteristics** (specialty, tier-like attributes, etc.) to predict adoption.

We use a **left merge** (file1 as the left/base table) so we do not lose any physician-quarter observations, and we
verify row/physician counts before and after to catch merge issues (duplicate keys inflating rows, physicians in
file1 missing from file2, etc.).

In [ ]:
rows_before = len(df1)
physicians_before = df1['physician_id'].nunique()

df_merged = df1.merge(df2, on='physician_id', how='left', validate='m:1')

rows_after = len(df_merged)
physicians_after = df_merged['physician_id'].nunique()

print(f"Rows before merge: {rows_before}")
print(f"Rows after merge:  {rows_after}")
print(f"Unique physicians before merge: {physicians_before}")
print(f"Unique physicians after merge:  {physicians_after}")
print(f"Row count changed unexpectedly: {rows_before != rows_after}")
print(f"Physicians in file1 with NO matching profile in file2: "
      f"{df_merged[static_cat_cols + static_num_cols].isna().all(axis=1).sum() if static_cat_cols+static_num_cols else 'n/a'}")

assert rows_before == rows_after, "Merge changed row count — indicates duplicate keys in file2 (should be 1 row/physician)."
print("\nMerge validated: 1 static profile row per physician, no row inflation.")


### Data Quality Checks

We check duplicates, data types, and physician-quarter completeness (each physician should have up to 10 quarters;
gaps are allowed but must be respected later when constructing lags and the next-quarter target).

In [ ]:
print("Duplicate physician-quarter rows:", df_merged.duplicated(subset=['physician_id','year_quarter']).sum())

quarters_per_physician = df_merged.groupby('physician_id')['year_quarter'].nunique()
print("\nDistribution of #quarters observed per physician:")
print(quarters_per_physician.value_counts().sort_index())

print("\nAll distinct year_quarter values present:")
print(sorted(df_merged['year_quarter'].unique()))

print("\nTarget (brand_prescribed) value counts:")
print(df_merged['brand_prescribed'].value_counts(dropna=False))


### Missing-Value Analysis

We compute missing count/percentage per feature. Per the project brief, we use **~20–30% missingness only as a
screening threshold** — we do NOT auto-drop; each high-missingness column is reviewed for business relevance and
whether "missing" itself might be meaningful (e.g., a physician with no digital-engagement data may simply not be
digitally active, which is itself informative) before any drop decision.

In [ ]:
missing_summary = pd.DataFrame({
    'missing_count': df_merged.isna().sum(),
    'missing_pct': (df_merged.isna().mean() * 100).round(2)
}).sort_values('missing_pct', ascending=False)

print(missing_summary)

SCREEN_THRESHOLD = 25  # % — screening only, not an auto-drop rule
high_missing = missing_summary[missing_summary['missing_pct'] > SCREEN_THRESHOLD]
print(f"\nColumns above the {SCREEN_THRESHOLD}% screening threshold (REVIEW, do not auto-drop):")
print(high_missing)

# Decision log — fill in / edit after reviewing high_missing above.
# Default stance: keep all columns; numeric NaNs are median-imputed and categorical NaNs are
# most-frequent-imputed later inside the sklearn Pipeline (fit on train folds only), which is
# a defensible default unless a specific column is found to be missing for a systematic,
# target-correlated reason (in which case an explicit "was_missing" flag would be considered).
dropped_features = []   # <-- document any column you decide to drop here, with a one-line reason as a comment
print("\nFeatures dropped after review:", dropped_features if dropped_features else "None — all features retained.")


## Exploratory Data Analysis (EDA)

Focused EDA covering dataset shape, distributions, target balance, quarterly coverage, and correlations —
enough to understand the data without producing unnecessary plots.

In [ ]:
print("Dataset dimensions:", df_merged.shape)
print("\nData types:\n", df_merged.dtypes.value_counts())

# Target distribution (row-level; NOT yet the modeling target — modeling target is defined later as next-quarter adoption)
plt.figure(figsize=(5,4))
df_merged['brand_prescribed'].value_counts().plot(kind='bar', color=['#4C72B0','#DD8452'])
plt.title("Row-level brand_prescribed distribution (current-quarter flag)")
plt.xlabel("brand_prescribed"); plt.ylabel("count")
plt.tight_layout(); plt.show()

# Quarterly distribution
plt.figure(figsize=(8,4))
df_merged['year_quarter'].value_counts().sort_index().plot(kind='bar', color='#55A868')
plt.title("Row count per year_quarter")
plt.tight_layout(); plt.show()


In [ ]:
# Numeric feature distributions (behavioral/activity columns from file1)
numeric_behavior_cols = [
    'total_representative_visits','total_sample_dropped','saving_cards_dropped','vouchers_dropped',
    'total_seminar_as_attendee','total_seminar_as_speaker',
    'total_prescriptions_for_indication1','total_prescriptions_for_indication2','total_prescriptions_for_indication3',
    'total_patient_with_commercial_insurance_plan','total_patient_with_medicare_insurance_plan',
    'total_patient_with_medicaid_insurance_plan',
    'brand_web_impressions','brand_ehr_impressions','brand_enews_impressions','brand_mobile_impressions',
    'brand_organic_web_visits','brand_paidsearch_visits','total_competitor_prescription','new_prescriptions'
]
numeric_behavior_cols = [c for c in numeric_behavior_cols if c in df_merged.columns]

df_merged[numeric_behavior_cols].describe().T

fig, axes = plt.subplots(5, 4, figsize=(18,16))
for ax, col in zip(axes.flatten(), numeric_behavior_cols):
    sns.histplot(df_merged[col].dropna(), ax=ax, bins=30, color='#4C72B0')
    ax.set_title(col, fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# Categorical feature distributions
categorical_cols_file1 = [c for c in ['physician_hospital_affiliation','physician_in_group_practice',
                                       'physician_value_tier'] if c in df_merged.columns]
for col in categorical_cols_file1 + static_cat_cols:
    print(f"\n--- {col} ---")
    print(df_merged[col].value_counts(dropna=False))

# Outlier check (IQR-based flag) on key activity variables — informational, not auto-removed
for col in ['total_representative_visits','total_sample_dropped']:
    if col in df_merged.columns:
        q1, q3 = df_merged[col].quantile([0.25,0.75])
        iqr = q3 - q1
        n_outliers = ((df_merged[col] < q1-1.5*iqr) | (df_merged[col] > q3+1.5*iqr)).sum()
        print(f"{col}: {n_outliers} potential IQR outliers (kept — activity counts are legitimately skewed)")


In [ ]:
# Correlation analysis among numeric variables
corr_cols = numeric_behavior_cols + static_num_cols
corr = df_merged[corr_cols].corr()

plt.figure(figsize=(14,10))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title("Correlation matrix — numeric features")
plt.tight_layout(); plt.show()

# Flag pairs with |corr| > 0.8 for later multicollinearity review
high_corr_pairs = []
for i in range(len(corr_cols)):
    for j in range(i+1, len(corr_cols)):
        c = corr.iloc[i,j]
        if abs(c) > 0.8:
            high_corr_pairs.append((corr_cols[i], corr_cols[j], round(c,3)))
print("Highly correlated pairs (|r|>0.8):", high_corr_pairs)

# Leakage-relevance check for the ambiguous `new_prescriptions` column:
# correlate it against the SAME-quarter brand_prescribed flag to gauge whether it looks brand-specific.
if 'new_prescriptions' in df_merged.columns:
    same_q_corr = df_merged[['new_prescriptions','brand_prescribed']].corr().iloc[0,1]
    print(f"\nCorrelation of new_prescriptions with SAME-quarter brand_prescribed: {same_q_corr:.3f}")
    print("If this is very high, treat new_prescriptions as likely brand-specific and higher leakage risk when lagged near the adoption quarter.")


## Create a Proper Time Variable

`year_quarter` is stored as e.g. `201502` (YYYY + single-digit quarter, confirmed from the data: values observed are
`201502, 201503, 201504, 201601, 201602, 201603, 201604, 201701, 201702, 201703` — i.e., quarters 2/2015 through
3/2017, 10 consecutive quarters). String-sorting this field would work by coincidence here, but we build an explicit
`time_index` so consecutive quarters always differ by exactly 1 — this is what makes lag/target construction safe.

In [ ]:
def create_time_index(df, quarter_col='year_quarter'):
    """Parse a YYYYQ-style integer quarter code into year, quarter_number, and a chronological time_index."""
    df = df.copy()
    df['year'] = df[quarter_col] // 10
    df['quarter_number'] = df[quarter_col] % 10
    assert df['quarter_number'].between(1,4).all(), "Unexpected quarter_number outside 1-4 — check year_quarter format."
    # time_index: consecutive integer, +1 per quarter, monotonic across years
    df['time_index'] = (df['year'] - df['year'].min()) * 4 + df['quarter_number']
    return df

df_merged = create_time_index(df_merged)
df_merged = df_merged.sort_values(['physician_id','time_index']).reset_index(drop=True)

# Verify ordering: time_index deltas within each physician should be positive and there should be no duplicates
chk = df_merged.groupby('physician_id')['time_index'].apply(lambda s: s.is_monotonic_increasing).all()
print("time_index strictly increasing within every physician:", chk)

time_lookup = df_merged[['year_quarter','year','quarter_number','time_index']].drop_duplicates().sort_values('time_index')
print(time_lookup)


## Target Definition: Next-Quarter Adoption

For a physician who is a **non-adopter through quarter t**, we predict `brand_prescribed` at quarter **t+1**:

```
X_t (info available through quarter t)  ->  Y_(t+1) = adoption in the following quarter
```

We also enforce **non-adopter logic**: once a physician's first adoption event occurs, later quarters are no longer
valid "new adoption" prediction rows (that physician is already a customer, not a targeting candidate).

And we enforce **quarter continuity**: we only create the target where `time_index(t+1) == time_index(t) + 1` — i.e.,
we never assume the "next row" for a physician is really the next quarter (a physician could have gaps).

In [ ]:
def create_target(df, id_col='physician_id', time_col='time_index', target_col='brand_prescribed'):
    """
    Build the next-quarter adoption target with:
      - quarter-continuity verification (only valid consecutive-quarter pairs get a target)
      - current-non-adopter filtering (cumulative adoption history excludes post-adoption rows)
    Returns df with columns: target_next_quarter (nullable), is_eligible_non_adopter
    """
    df = df.sort_values([id_col, time_col]).copy()

    # cumulative adoption flag: has this physician EVER adopted up to and including this quarter?
    df['cum_adopted_through_t'] = df.groupby(id_col)[target_col].cummax()

    # eligible as a "current non-adopter" observation only if they had NOT adopted by quarter t
    # (i.e., cum_adopted_through_t at the PRIOR row was 0; for the very first row of a physician, prior=0 by definition)
    prior_cum = df.groupby(id_col)['cum_adopted_through_t'].shift(1).fillna(0)
    df['is_eligible_non_adopter'] = (prior_cum == 0)

    # next quarter's target value and next quarter's time_index, within physician
    df['next_target_raw'] = df.groupby(id_col)[target_col].shift(-1)
    df['next_time_index']  = df.groupby(id_col)[time_col].shift(-1)

    # a target is VALID only if the next row truly is the immediately-following quarter
    df['quarter_continuity_ok'] = (df['next_time_index'] == df[time_col] + 1)

    df['target_next_quarter'] = np.where(df['quarter_continuity_ok'], df['next_target_raw'], np.nan)

    return df

df_merged = create_target(df_merged)
print(df_merged[['physician_id','time_index','brand_prescribed','cum_adopted_through_t',
                  'is_eligible_non_adopter','next_time_index','quarter_continuity_ok','target_next_quarter']].head(15))


### Filter to Valid Current-Non-Adopter → Next-Quarter Observations

A row is a valid **modeling observation** only if:
1. `is_eligible_non_adopter` — the physician had not adopted before/at quarter t, **and**
2. `quarter_continuity_ok` — quarter t+1 truly exists and is consecutive, **and**
3. `target_next_quarter` is not null.

This automatically implements the business rule from the brief: for a physician `[0,0,0,1,1,1]`, only
`Q1->Q2, Q2->Q3, Q3->Q4` are valid observations; `Q4->Q5, Q5->Q6` are excluded because the physician had already
adopted by Q4.

In [ ]:
labeled_df = df_merged[
    df_merged['is_eligible_non_adopter'] &
    df_merged['quarter_continuity_ok'] &
    df_merged['target_next_quarter'].notna()
].copy()

labeled_df['target_next_quarter'] = labeled_df['target_next_quarter'].astype(int)

print("Total merged rows:", len(df_merged))
print("Valid labeled (non-adopter -> next-quarter) modeling rows:", len(labeled_df))
print("\nTarget distribution in labeled_df:")
print(labeled_df['target_next_quarter'].value_counts(normalize=True).round(4))

# sanity check: no row in labeled_df should itself already have brand_prescribed==1 at quarter t
assert (labeled_df['brand_prescribed'] == 0).all(), "Found rows where the physician already adopted at quarter t — non-adopter filter failed."
print("\nSanity check passed: every labeled row is a genuine current non-adopter at quarter t.")


## Lag-1 and Lag-2 Feature Engineering

Per the project brief, we use **only Lag-1 (previous quarter) and Lag-2 (two quarters ago)** — no Lag-3/Lag-4.
Lags are created **per physician**, after sorting by `time_index`, using `groupby(physician_id).shift()`. This
guarantees no cross-physician leakage and (combined with the continuity check we already ran) that a lag only
reflects real historical values, never an imputed/wrong-quarter value silently mislabeled as "previous quarter".

Note: `brand_prescribed` itself is **excluded** from the lag feature set — since every labeled row is, by
construction, a non-adopter through quarter t, `brand_prescribed_lag1`/`lag2` would be structurally constant (0)
and carries no signal; including it would be a wasted/misleading feature, not true leakage, but we drop it for
cleanliness.

In [ ]:
def create_lag_features(df, id_col='physician_id', time_col='time_index', cols=None, n_lags=2):
    """Create lag1..lag_n features for the given numeric columns, strictly within each physician's own timeline."""
    df = df.sort_values([id_col, time_col]).copy()
    lag_cols_created = []
    for col in cols:
        for lag in range(1, n_lags+1):
            new_col = f"{col}_lag{lag}"
            df[new_col] = df.groupby(id_col)[col].shift(lag)
            lag_cols_created.append(new_col)
    return df, lag_cols_created

labeled_df, lag_cols = create_lag_features(labeled_df, cols=numeric_behavior_cols, n_lags=2)
print(f"Created {len(lag_cols)} lag features from {len(numeric_behavior_cols)} numeric behavioral columns.")
print(lag_cols[:6], "...")


## Lag-1 for Categorical Time-Varying Columns

`physician_hospital_affiliation`, `physician_in_group_practice`, and `physician_value_tier` can change quarter to
quarter, so we also carry forward their **Lag-1 value** (their state as of the prediction point) as a feature — but
we do **not** sum or average categorical variables, only take the lagged categorical value itself.

In [ ]:
categorical_lag1_cols = []
for col in categorical_cols_file1:
    new_col = f"{col}_lag1"
    labeled_df[new_col] = labeled_df.groupby('physician_id')[col].shift(1)
    categorical_lag1_cols.append(new_col)

print("Categorical Lag-1 columns created:", categorical_lag1_cols)


## Derived Features from Lag-1 / Lag-2

For count/activity-style numeric variables, we create business-meaningful derived features:

- **2-quarter average** — smoothed recent activity level.
- **2-quarter sum** — total recent activity volume.
- **Change (Lag1 − Lag2)** — momentum: is engagement/prescribing trending up or down?

These are only created for count/activity variables (visits, samples, impressions, prescriptions, etc.) — not for
static categorical variables, where sums/averages are meaningless.

In [ ]:
def create_derived_features(df, base_cols):
    """For each base column with existing _lag1/_lag2 features, add 2q-avg, 2q-sum, and change."""
    df = df.copy()
    derived_cols = []
    for col in base_cols:
        l1, l2 = f"{col}_lag1", f"{col}_lag2"
        if l1 in df.columns and l2 in df.columns:
            df[f"{col}_avg_2q"]    = (df[l1] + df[l2]) / 2
            df[f"{col}_sum_2q"]    = df[l1] + df[l2]
            df[f"{col}_change_2q"] = df[l1] - df[l2]
            derived_cols += [f"{col}_avg_2q", f"{col}_sum_2q", f"{col}_change_2q"]
    return df, derived_cols

labeled_df, derived_cols = create_derived_features(labeled_df, numeric_behavior_cols)
print(f"Created {len(derived_cols)} derived features.")


## Static Physician Profile Features

Static features from `Input_data_file2` were already merged in (Cell 6) and are physician-level (not
quarter-varying), so they need no lagging — they describe the physician's baseline characteristics.

In [ ]:
print("Static categorical features (from file2):", static_cat_cols)
print("Static numeric features (from file2):", static_num_cols)

# quick missingness re-check after all the row filtering
static_missing = labeled_df[static_cat_cols + static_num_cols].isna().mean().round(3) * 100
print("\nMissing % in static features within labeled_df:\n", static_missing)


## Correlation & Multicollinearity Diagnostics

**What multicollinearity is:** when two or more predictors are highly linearly correlated, making it hard for a
linear model to attribute effect to any one of them individually — coefficients become unstable and hard to
interpret, though predictive accuracy is often still fine.

**Why it matters for Logistic Regression:** LR coefficients are the primary interpretability output; multicollinearity
inflates their variance and can flip signs, undermining the "why" behind a prediction.

**Why VIF matters less for Random Forest / XGBoost:** tree-based models split on one feature at a time and don't
estimate linear coefficients, so correlated features mainly cause *redundant* splits/diluted importance, not unstable
predictions — they remain robust to multicollinearity for prediction purposes (though it can muddy importance ranking).

We therefore compute VIF as a **diagnostic for Logistic Regression**, and do **not** treat high correlation as an
automatic feature-removal rule for the tree models — decisions are business/interpretability-driven, documented, not
automatic.

In [ ]:
numeric_feature_cols = lag_cols + derived_cols + static_num_cols
numeric_feature_cols = [c for c in numeric_feature_cols if c in labeled_df.columns]

corr_feat = labeled_df[numeric_feature_cols].corr()
plt.figure(figsize=(16,12))
sns.heatmap(corr_feat, cmap='coolwarm', center=0)
plt.title("Correlation matrix — engineered numeric features")
plt.tight_layout(); plt.show()

high_corr_feat_pairs = []
for i in range(len(numeric_feature_cols)):
    for j in range(i+1, len(numeric_feature_cols)):
        c = corr_feat.iloc[i,j]
        if abs(c) > 0.85:
            high_corr_feat_pairs.append((numeric_feature_cols[i], numeric_feature_cols[j], round(c,3)))
print(f"{len(high_corr_feat_pairs)} highly correlated engineered-feature pairs (|r|>0.85):")
for p in high_corr_feat_pairs[:20]:
    print(p)
print("\nNote: sum_2q, avg_2q, and lag1/lag2 of the SAME base variable are expected to correlate highly by construction. "
      "This is not automatically removed (tree models handle it fine); Logistic Regression relies on the VIF check below instead.")


In [ ]:
# VIF diagnostic (Logistic Regression only) — computed on a complete-case, median-imputed sample for feasibility
vif_sample = labeled_df[numeric_feature_cols].fillna(labeled_df[numeric_feature_cols].median())
vif_sample = vif_sample.loc[:, vif_sample.std() > 0]   # drop constant columns which break VIF

vif_data = pd.DataFrame()
vif_data["feature"] = vif_sample.columns
vif_data["VIF"] = [variance_inflation_factor(vif_sample.values, i) for i in range(vif_sample.shape[1])]
vif_data = vif_data.sort_values("VIF", ascending=False)
print(vif_data.head(25))
print("\nRule of thumb: VIF > 10 indicates problematic multicollinearity for Logistic Regression coefficient "
      "interpretation. This is a diagnostic, not an automatic removal rule — document any feature you choose to "
      "drop and why below.")


## Define Final Feature Groups

In [ ]:
FINAL_NUMERIC_FEATURES = numeric_feature_cols
FINAL_CATEGORICAL_FEATURES = categorical_lag1_cols + static_cat_cols
FINAL_CATEGORICAL_FEATURES = [c for c in FINAL_CATEGORICAL_FEATURES if c in labeled_df.columns]

ID_COLS = ['physician_id']
TIME_COLS = ['time_index','year','quarter_number','year_quarter']
TARGET_COL = 'target_next_quarter'

ALL_FEATURES = FINAL_NUMERIC_FEATURES + FINAL_CATEGORICAL_FEATURES

print(f"Numeric features: {len(FINAL_NUMERIC_FEATURES)}")
print(f"Categorical features: {len(FINAL_CATEGORICAL_FEATURES)}")
print(f"Total features: {len(ALL_FEATURES)}")
print("\nColumns explicitly EXCLUDED from modeling (leakage/ID/current-quarter/raw target):")
excluded = set(df_merged.columns) - set(ALL_FEATURES) - set(ID_COLS) - {TARGET_COL}
print(sorted(excluded))


## Chronological 80% / 20% Split

We split the **quarters**, not the rows, chronologically: the earliest ~80% of `time_index` values become the
development set, the latest ~20% become the untouched final historical test set. This mirrors PAST → FUTURE and is
never a random row split (random splitting would let future information leak into training via shared physicians'
neighboring quarters).

In [ ]:
all_time_indices = sorted(labeled_df['time_index'].unique())
n_quarters = len(all_time_indices)
split_point = int(np.floor(n_quarters * 0.8))
dev_quarters  = all_time_indices[:split_point]
test_quarters = all_time_indices[split_point:]

print("All available (t) time_index values with a valid target:", all_time_indices)
print("Development (80%) quarters:", dev_quarters)
print("Final historical TEST (20%) quarters:", test_quarters)

dev_df  = labeled_df[labeled_df['time_index'].isin(dev_quarters)].copy()
test_df = labeled_df[labeled_df['time_index'].isin(test_quarters)].copy()

print(f"\nDevelopment rows: {len(dev_df)}  |  Final historical test rows: {len(test_df)}")
print("Development target rate:", dev_df[TARGET_COL].mean().round(4))
print("Final test target rate:", test_df[TARGET_COL].mean().round(4))


## Temporal Cross-Validation Within the 80% Development Set

Using only the development quarters, we build rolling/forward folds: each fold trains on all quarters strictly
before a validation quarter, and validates on that single later quarter. A physician can appear in both train and
validation (their earlier-quarter row trains, their later-quarter row validates) — that mirrors the real business
setting (learn from history, predict the future) — but no *future* quarter's information ever enters an earlier
fold's training features, because lag features were built using shift() at row-construction time, before any split.

In [ ]:
def create_temporal_folds(df, time_col='time_index', min_train_quarters=3):
    """Rolling-origin folds: train on all quarters < v, validate on quarter v, for each v with enough history."""
    quarters = sorted(df[time_col].unique())
    folds = []
    for i in range(min_train_quarters, len(quarters)):
        val_q = quarters[i]
        train_qs = quarters[:i]
        train_idx = df[df[time_col].isin(train_qs)].index
        val_idx   = df[df[time_col] == val_q].index
        folds.append((train_qs, val_q, train_idx, val_idx))
    return folds

folds = create_temporal_folds(dev_df, min_train_quarters=3)
for train_qs, val_q, tr_idx, va_idx in folds:
    print(f"Train quarters {train_qs} (n={len(tr_idx)})  ->  Validate quarter {val_q} (n={len(va_idx)})")


## Preprocessing Pipeline

Numeric features: median imputation + standard scaling. Categorical features: most-frequent imputation + one-hot
encoding. Built with `Pipeline` + `ColumnTransformer` so that **all fitting happens only on each fold's training
data** — never on validation, the 20% historical test set, or the Q11 population.

In [ ]:
def build_preprocessor(numeric_cols, categorical_cols):
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])
    preprocessor = ColumnTransformer([
        ('num', numeric_pipe, numeric_cols),
        ('cat', categorical_pipe, categorical_cols)
    ])
    return preprocessor

preprocessor = build_preprocessor(FINAL_NUMERIC_FEATURES, FINAL_CATEGORICAL_FEATURES)
print(preprocessor)


## Evaluation Functions: Lift@20% and the Full Metric Suite

**Lift@20%** = (adoption rate among the top-20%-ranked physicians) / (overall adoption rate in that fold). It answers
the business question directly: *"if the sales team can only reach the top 20% we recommend, how much better is that
than reaching a random 20%?"* This is the primary metric because sales capacity is fixed — ranking quality in the
actionable top slice matters more than overall discrimination.

- **Precision**: of physicians flagged as likely adopters, how many actually adopt?
- **Recall**: of all physicians who actually adopt, how many did we correctly flag?
- **F1**: balance of precision and recall.
- **ROC-AUC**: overall ability to rank adopters above non-adopters across all thresholds.
- **PR-AUC**: focuses on the positive (adopter) class — more informative than ROC-AUC when adoption is rare.

In [ ]:
def calculate_lift_at_k(y_true, y_scores, k=0.2):
    """Lift at top-k fraction: (positive rate in top-k) / (overall positive rate)."""
    n = len(y_true)
    n_top = max(1, int(np.ceil(n * k)))
    order = np.argsort(-np.asarray(y_scores))
    top_idx = order[:n_top]
    top_rate = np.asarray(y_true)[top_idx].mean()
    overall_rate = np.asarray(y_true).mean()
    if overall_rate == 0:
        return np.nan
    return top_rate / overall_rate

def evaluate_model(y_true, y_scores, threshold=0.5):
    y_pred = (np.asarray(y_scores) >= threshold).astype(int)
    metrics = {
        'roc_auc': roc_auc_score(y_true, y_scores),
        'pr_auc': average_precision_score(y_true, y_scores),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'lift_at_20': calculate_lift_at_k(y_true, y_scores, k=0.2)
    }
    return metrics

# quick sanity example matching the brief: overall 10%, top20% 25% -> lift 2.5
_y = [1]*10 + [0]*90
_s = list(np.linspace(1,0,10)) + list(np.linspace(0.9,0,90))
print("Sanity check lift value (illustrative, not exact due to random tie ordering):",
      round(calculate_lift_at_k(_y, _s, 0.2), 2))


## Train & Evaluate: Logistic Regression (Temporal CV)

In [ ]:
def run_temporal_cv(model_builder, df, folds, numeric_cols, categorical_cols, target_col, model_name):
    fold_results = []
    for train_qs, val_q, tr_idx, va_idx in folds:
        X_train = df.loc[tr_idx, numeric_cols + categorical_cols]
        y_train = df.loc[tr_idx, target_col]
        X_val   = df.loc[va_idx, numeric_cols + categorical_cols]
        y_val   = df.loc[va_idx, target_col]

        preproc = build_preprocessor(numeric_cols, categorical_cols)   # fresh preprocessor fit ONLY on this fold's train
        model = model_builder()
        pipe = Pipeline([('preprocessor', preproc), ('model', model)])
        pipe.fit(X_train, y_train)

        y_scores = pipe.predict_proba(X_val)[:, 1]
        m = evaluate_model(y_val, y_scores)
        m.update({'model': model_name, 'val_quarter': val_q, 'n_train': len(tr_idx), 'n_val': len(va_idx)})
        fold_results.append(m)
    return pd.DataFrame(fold_results)

logreg_builder = lambda: LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
logreg_cv_results = run_temporal_cv(logreg_builder, dev_df, folds, FINAL_NUMERIC_FEATURES,
                                     FINAL_CATEGORICAL_FEATURES, TARGET_COL, 'LogisticRegression')
print(logreg_cv_results)


## Train & Evaluate: Random Forest (Temporal CV)

In [ ]:
rf_builder = lambda: RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                             class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf_cv_results = run_temporal_cv(rf_builder, dev_df, folds, FINAL_NUMERIC_FEATURES,
                                 FINAL_CATEGORICAL_FEATURES, TARGET_COL, 'RandomForest')
print(rf_cv_results)


## Train & Evaluate: XGBoost (Temporal CV)

Class imbalance for XGBoost is handled via `scale_pos_weight` **only if** the observed target imbalance in the
development data justifies it (computed from the data, not assumed).

In [ ]:
if XGB_AVAILABLE:
    pos_rate = dev_df[TARGET_COL].mean()
    scale_pos_weight = (1 - pos_rate) / pos_rate if pos_rate > 0 else 1.0
    print(f"Observed positive rate in dev_df: {pos_rate:.4f}  ->  scale_pos_weight = {scale_pos_weight:.2f}")

    xgb_builder = lambda: XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
    )
    xgb_cv_results = run_temporal_cv(xgb_builder, dev_df, folds, FINAL_NUMERIC_FEATURES,
                                      FINAL_CATEGORICAL_FEATURES, TARGET_COL, 'XGBoost')
    print(xgb_cv_results)
else:
    xgb_cv_results = pd.DataFrame()
    print("Skipped — xgboost not installed.")


## Compare Temporal CV Results Across Models

In [ ]:
all_cv_results = pd.concat([logreg_cv_results, rf_cv_results, xgb_cv_results], ignore_index=True)

summary = all_cv_results.groupby('model').agg(
    mean_roc_auc=('roc_auc','mean'), std_roc_auc=('roc_auc','std'),
    mean_pr_auc=('pr_auc','mean'), std_pr_auc=('pr_auc','std'),
    mean_precision=('precision','mean'), mean_recall=('recall','mean'),
    mean_f1=('f1','mean'),
    mean_lift_at_20=('lift_at_20','mean'), std_lift_at_20=('lift_at_20','std')
).reset_index().sort_values('mean_lift_at_20', ascending=False)

print(summary)
summary.to_csv('model_comparison.csv', index=False)
all_cv_results.to_csv('validation_results.csv', index=False)


## Select Best Model

Model selection is driven **primarily by mean Lift@20%** across temporal folds (the direct business objective —
ranking quality in the actionable top slice), with PR-AUC/ROC-AUC/Precision/Recall/F1 as secondary tie-breakers and
sanity checks. We deliberately do NOT select on accuracy. The `summary` table above is sorted by `mean_lift_at_20`
descending — the top row is the selected model.

In [ ]:
BEST_MODEL_NAME = summary.iloc[0]['model']
print(f"Selected model (highest mean Lift@20% in temporal CV): {BEST_MODEL_NAME}")
print(summary.iloc[0])

model_builders = {'LogisticRegression': logreg_builder, 'RandomForest': rf_builder}
if XGB_AVAILABLE:
    model_builders['XGBoost'] = xgb_builder

best_model_builder = model_builders[BEST_MODEL_NAME]


## Retrain Selected Model on the Complete 80% Development Data

In [ ]:
X_dev = dev_df[FINAL_NUMERIC_FEATURES + FINAL_CATEGORICAL_FEATURES]
y_dev = dev_df[TARGET_COL]

dev_preprocessor = build_preprocessor(FINAL_NUMERIC_FEATURES, FINAL_CATEGORICAL_FEATURES)
dev_pipeline = Pipeline([('preprocessor', dev_preprocessor), ('model', best_model_builder())])
dev_pipeline.fit(X_dev, y_dev)
print(f"{BEST_MODEL_NAME} retrained on the full 80% development set ({len(X_dev)} rows).")


## FINAL HISTORICAL TEST PERFORMANCE — Evaluate on the Untouched 20%

This is the **first and only** time the final 20% historical test set is used. It was not used for feature
selection, hyperparameter tuning, model selection, or preprocessing fitting.

In [ ]:
X_test_hist = test_df[FINAL_NUMERIC_FEATURES + FINAL_CATEGORICAL_FEATURES]
y_test_hist = test_df[TARGET_COL]

test_scores = dev_pipeline.predict_proba(X_test_hist)[:, 1]
final_hist_metrics = evaluate_model(y_test_hist, test_scores)

print("=== FINAL HISTORICAL TEST PERFORMANCE (untouched 20%, out-of-sample) ===")
for k, v in final_hist_metrics.items():
    print(f"{k}: {v:.4f}")

cm = confusion_matrix(y_test_hist, (test_scores >= 0.5).astype(int))
print("\nConfusion matrix (threshold=0.5):\n", cm)

pd.DataFrame([final_hist_metrics]).to_csv('final_historical_test_metrics.csv', index=False)


## Feature Importance / Model Interpretation

We report the model's native feature importance (or coefficients for Logistic Regression) plus a
model-agnostic **permutation importance** computed on the held-out 20% historical test set, for a second, less
model-biased view of what's driving predictions.

In [ ]:
# Recover feature names after one-hot encoding
ohe = dev_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['encoder']
cat_feature_names = list(ohe.get_feature_names_out(FINAL_CATEGORICAL_FEATURES)) if FINAL_CATEGORICAL_FEATURES else []
all_feature_names = FINAL_NUMERIC_FEATURES + cat_feature_names

model_step = dev_pipeline.named_steps['model']
if hasattr(model_step, 'feature_importances_'):
    importances = model_step.feature_importances_
elif hasattr(model_step, 'coef_'):
    importances = np.abs(model_step.coef_[0])
else:
    importances = np.zeros(len(all_feature_names))

native_importance_df = pd.DataFrame({'feature': all_feature_names, 'native_importance': importances}) \
    .sort_values('native_importance', ascending=False)
print("Top 20 features by native model importance:")
print(native_importance_df.head(20))

# Permutation importance on the historical 20% test set
perm_result = permutation_importance(dev_pipeline, X_test_hist, y_test_hist,
                                      n_repeats=10, random_state=RANDOM_STATE, scoring='roc_auc', n_jobs=-1)
perm_importance_df = pd.DataFrame({
    'feature': X_test_hist.columns,
    'perm_importance_mean': perm_result.importances_mean,
    'perm_importance_std': perm_result.importances_std
}).sort_values('perm_importance_mean', ascending=False)

print("\nTop 20 features by permutation importance (ROC-AUC drop):")
print(perm_importance_df.head(20))

feature_importance_export = native_importance_df.merge(
    perm_importance_df.rename(columns={'feature':'feature'}), on='feature', how='outer'
)
feature_importance_export.to_csv('feature_importance.csv', index=False)


## Retrain Final Model on ALL Historical Labeled Data (Through Q10)

Having validated methodology and performance on the untouched 20%, we now retrain the **same selected model type**
on the entire historical labeled dataset (both the 80% and the 20%) to squeeze out maximum information before
scoring the real Quarter-11 population — this is standard practice once the evaluation has already happened.

In [ ]:
X_all_hist = labeled_df[FINAL_NUMERIC_FEATURES + FINAL_CATEGORICAL_FEATURES]
y_all_hist = labeled_df[TARGET_COL]

final_preprocessor = build_preprocessor(FINAL_NUMERIC_FEATURES, FINAL_CATEGORICAL_FEATURES)
final_pipeline = Pipeline([('preprocessor', final_preprocessor), ('model', best_model_builder())])
final_pipeline.fit(X_all_hist, y_all_hist)
print(f"Final {BEST_MODEL_NAME} retrained on all {len(X_all_hist)} historical labeled rows (Q1-Q10).")


## Prepare the ~1,500 Quarter-11 Test Physicians

For each Q11 test physician we need: Lag-1 = Q10 value, Lag-2 = Q9 value, the same derived features, and their
static profile — built with the exact same functions used during training, applied to their most recent two
historical quarters from `Input_data_file1`.

In [ ]:
# Confirm the test population are indeed physicians who never adopted the brand across Q1-Q10
test_physician_ids = df_test['physician_id'].unique()
history_for_test = df_merged[df_merged['physician_id'].isin(test_physician_ids)].copy()

never_adopted_check = history_for_test.groupby('physician_id')['brand_prescribed'].max()
n_unexpected_adopters = (never_adopted_check == 1).sum()
print(f"Q11 test physicians found in historical data: {history_for_test['physician_id'].nunique()} / {len(test_physician_ids)}")
print(f"Of these, physicians with brand_prescribed==1 at any point in Q1-Q10 (unexpected): {n_unexpected_adopters}")
if n_unexpected_adopters > 0:
    print("FLAG: some 'test' physicians show prior adoption in file1 — review before scoring (should be non-adopters by definition).")


In [ ]:
# Latest quarter (Q10) = the prediction anchor point; Lag-1 = Q10, Lag-2 = Q9
max_time_index = df_merged['time_index'].max()
print("Max time_index in historical data (this is 'Q10' / the anchor quarter for Q11 prediction):", max_time_index)

latest_row = (history_for_test[history_for_test['time_index'] == max_time_index]
              .drop_duplicates(subset='physician_id', keep='last'))

print(f"Q11 test physicians with a Q10 (latest quarter) record available: {latest_row['physician_id'].nunique()} / {len(test_physician_ids)}")
missing_latest = set(test_physician_ids) - set(latest_row['physician_id'])
if missing_latest:
    print(f"FLAG: {len(missing_latest)} test physicians have NO Q10 record in Input_data_file1 — "
          f"cannot build Lag-1 for them without further info. Example IDs: {list(missing_latest)[:5]}")


In [ ]:
# Build Lag-1/Lag-2 + derived features for the Q11 population using the SAME per-physician shift logic,
# then take each physician's row as of quarter time_index == max_time_index (their "current t" for predicting t+1=Q11).
history_for_test_sorted = history_for_test.sort_values(['physician_id','time_index'])
history_for_test_sorted, test_lag_cols = create_lag_features(history_for_test_sorted, cols=numeric_behavior_cols, n_lags=2)
history_for_test_sorted, test_derived_cols = create_derived_features(history_for_test_sorted, numeric_behavior_cols)

for col in categorical_cols_file1:
    history_for_test_sorted[f"{col}_lag1"] = history_for_test_sorted.groupby('physician_id')[col].shift(1)

q11_features_df = (history_for_test_sorted[history_for_test_sorted['time_index'] == max_time_index]
                    .drop_duplicates(subset='physician_id', keep='last').copy())

q11_features_df = q11_features_df[q11_features_df['physician_id'].isin(test_physician_ids)]
print("Q11 feature rows ready:", q11_features_df.shape)
print("Missing values in Lag-1 (should be 0, since these physicians have a full Q1-Q10 history):",
      q11_features_df[[c for c in test_lag_cols if c.endswith('_lag1')]].isna().sum().sum())


## Apply the Preprocessing Pipeline and Generate Q11 Probabilities

In [ ]:
missing_feature_cols = [c for c in (FINAL_NUMERIC_FEATURES + FINAL_CATEGORICAL_FEATURES) if c not in q11_features_df.columns]
if missing_feature_cols:
    print("FLAG - feature columns missing from the Q11 build (will error below, review upstream):", missing_feature_cols)

X_q11 = q11_features_df[FINAL_NUMERIC_FEATURES + FINAL_CATEGORICAL_FEATURES]
q11_probabilities = final_pipeline.predict_proba(X_q11)[:, 1]

q11_features_df['predicted_adoption_probability'] = q11_probabilities
print(q11_features_df[['physician_id','predicted_adoption_probability']].describe())


## Rank Physicians and Create High / Medium / Low Priority Segments

Top 20% -> High Priority, next 30% -> Medium Priority, bottom 50% -> Low Priority — sorted by predicted probability
descending, matching the business's limited-sales-capacity prioritization scheme.

In [ ]:
final_predictions = (q11_features_df[['physician_id','predicted_adoption_probability']]
                      .sort_values('predicted_adoption_probability', ascending=False)
                      .reset_index(drop=True))
final_predictions['rank'] = final_predictions.index + 1

n = len(final_predictions)
top20_cut = int(np.ceil(n * 0.2))
next30_cut = int(np.ceil(n * 0.5))

def assign_group(rank):
    if rank <= top20_cut:
        return 'High Priority'
    elif rank <= next30_cut:
        return 'Medium Priority'
    else:
        return 'Low Priority'

final_predictions['target_group'] = final_predictions['rank'].apply(assign_group)

print(final_predictions['target_group'].value_counts())
print(final_predictions.head(20))


## Export Final Output Files

In [ ]:
final_predictions.to_csv('physician_adoption_predictions.csv', index=False)
# model_comparison.csv, validation_results.csv, feature_importance.csv already written above

print("Files written:")
print(" - physician_adoption_predictions.csv  (physician_id, predicted_adoption_probability, rank, target_group)")
print(" - model_comparison.csv                (temporal CV summary per model)")
print(" - validation_results.csv              (fold-level results)")
print(" - feature_importance.csv              (native + permutation importance)")


## Final Business Interpretation

The model predicts the probability that each currently non-adopting physician will adopt drug XYZ in the following
quarter. Historical physician behavior is captured using Lag-1 and Lag-2 features (previous two quarters of rep
visits, samples, digital engagement, prescribing volume, etc.) together with static physician characteristics. The
model was developed using **chronological temporal cross-validation** inside an 80% development window so that no
future information could leak into training, then evaluated **once** on a completely untouched final 20% of
historical data to obtain an honest out-of-sample read on performance. After that evaluation, the model was
retrained on all historical data through Q10 and used to score the ~1,500 real Quarter-11 non-adopting physicians.
Physicians are ranked by predicted adoption probability, and — because sales capacity is limited — the **top 20%**
are flagged High Priority. **Lift@20%** quantifies how much more effective this targeted outreach is versus
targeting a random 20% of the same population.

**⚠️ Reminder:** Q11 physicians' true outcomes are unknown, so no accuracy/ROC-AUC/Lift metric is (or should be)
computed for them — only probability, rank, and priority group, per the project brief.

## Data Leakage & Methodology Checklist

In [ ]:
checklist = {
    "Input 1 and Input 2 merged correctly (m:1, no row inflation)": rows_before == rows_after,
    "No unexpected duplicate physician-quarter rows": df_merged.duplicated(subset=['physician_id','year_quarter']).sum()==0,
    "Missing values analyzed": True,
    "Target represents NEXT-QUARTER adoption": True,
    "Only current non-adopters modeled": bool((labeled_df['brand_prescribed']==0).all()),
    "Post-adoption observations excluded": True,
    "Quarter continuity verified before target creation": True,
    "Lag-1 and Lag-2 created per-physician only": True,
    "No Lag-3/Lag-4 features created": not any('_lag3' in c or '_lag4' in c for c in labeled_df.columns),
    "Preprocessing fit only on training folds / dev set": True,
    "Historical data split chronologically 80/20 (not random)": True,
    "20% historical test set used only once, at the end": True,
    "Temporal CV performed only within the 80%": True,
    "Logistic Regression, Random Forest, XGBoost all evaluated": XGB_AVAILABLE,
    "Model selected primarily on mean Lift@20%": True,
    "Final model retrained on all Q1-Q10 data before Q11 scoring": True,
    "Q11 predictions use only information available through Q10": True,
    "No performance metrics computed for Q11 (outcomes unknown)": True,
}
for k, v in checklist.items():
    print(f"[{'x' if v else ' '}] {k}")
